# SAR Oceanography Algorithms

## Motivation

SAR ocean products are usually derived from calibrated radar backscatter rather than from radiance or brightness temperature. The central challenge is that many ocean processes can produce similar image patterns. Wind, waves, slicks, rain, currents, ships, and sea ice can all change the returned radar signal.

This notebook introduces a few common processing ideas used in SAR oceanography. The examples are deliberately simplified. They are meant to build intuition, not to replace operational algorithms.

## From Image Power to Calibrated Backscatter

Raw SAR imagery is not directly comparable across sensors, acquisition modes, and viewing geometries. Quantitative applications usually start with a calibrated measure of normalized radar cross section, commonly written $\sigma^0$.

A simplified calibration relationship is:

$$\sigma^0 = K I$$

where:

- $I$ is an image intensity value
- $K$ is a calibration factor
- $\sigma^0$ is the calibrated backscatter coefficient

Operational processing includes additional corrections for instrument effects, geometry, terrain or ocean projection, noise, and product type.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Toy calibration from image intensity to sigma0
image_intensity = np.array([50, 100, 200, 400, 800])
K = 2.5e-5
sigma0 = K * image_intensity
sigma0_db = 10 * np.log10(sigma0)

for I, s, db in zip(image_intensity, sigma0, sigma0_db):
    print(f'Intensity: {I:4.0f}  sigma0: {s:.4f}  sigma0_dB: {db:.2f} dB')

## Speckle

SAR images contain **speckle**, a grainy interference pattern caused by coherent radar imaging. Speckle is not simply sensor noise; it is a consequence of the many small scatterers within each image cell adding together with different phases.

Speckle makes SAR images look noisy and can complicate interpretation. Common approaches include multilooking, spatial filtering, temporal averaging, and object-based analysis. These methods reduce speckle but also reduce spatial detail.

In [ ]:
# Create a synthetic ocean backscatter field with multiplicative speckle
np.random.seed(4)
ny, nx = 160, 220
x = np.linspace(-3, 3, nx)
y = np.linspace(-2, 2, ny)
X, Y = np.meshgrid(x, y)

# Smooth background with a bright wind streak and a dark slick-like feature
background = 0.04 + 0.01*np.sin(2*X) + 0.005*np.cos(3*Y)
wind_streak = 0.025*np.exp(-((Y - 0.45*np.sin(1.5*X))**2)/0.03)
slick = -0.025*np.exp(-((X+1.0)**2/0.5 + (Y+0.3)**2/0.08))
sigma0_true = np.clip(background + wind_streak + slick, 0.002, None)

# Multiplicative speckle using a gamma distribution
looks = 1
speckle = np.random.gamma(shape=looks, scale=1/looks, size=sigma0_true.shape)
sigma0_speckled = sigma0_true * speckle

plt.figure(figsize=(8, 4))
plt.imshow(10*np.log10(sigma0_speckled), origin='lower')
plt.title('Synthetic SAR Backscatter with Speckle')
plt.xlabel('Range pixel')
plt.ylabel('Azimuth pixel')
cb = plt.colorbar()
cb.set_label('Backscatter (dB)')
plt.show()

In [ ]:
# A simple mean filter implemented with NumPy only
# For real analyses, use established SAR processing tools rather than this toy filter.
def mean_filter_3x3(arr):
    padded = np.pad(arr, 1, mode='edge')
    out = np.zeros_like(arr)
    for j in range(arr.shape[0]):
        for i in range(arr.shape[1]):
            out[j, i] = padded[j:j+3, i:i+3].mean()
    return out

sigma0_filtered = mean_filter_3x3(sigma0_speckled)

plt.figure(figsize=(8, 4))
plt.imshow(10*np.log10(sigma0_filtered), origin='lower')
plt.title('Synthetic SAR Backscatter after a 3x3 Mean Filter')
plt.xlabel('Range pixel')
plt.ylabel('Azimuth pixel')
cb = plt.colorbar()
cb.set_label('Backscatter (dB)')
plt.show()

## Wind Speed Retrieval: The Geophysical Model Function Idea

For ocean wind products, SAR backscatter is related to wind speed, wind direction relative to the radar look direction, incidence angle, polarization, and radar frequency. Operational wind retrievals use **geophysical model functions** and often require an external wind direction estimate.

A toy relationship might look like this:

$$\sigma^0_{dB} = a + bU - c(	heta - 30)$$

where:

- $U$ is wind speed
- $	heta$ is incidence angle
- $a$, $b$, and $c$ are empirical constants

This is not an operational model, but it illustrates why the same wind speed can produce different backscatter at different incidence angles.

In [ ]:
def toy_wind_backscatter(wind_speed, incidence_angle, a=-18, b=0.7, c=0.18):
    """Toy relationship between wind speed, incidence angle, and SAR backscatter."""
    return a + b*wind_speed - c*(incidence_angle - 30)

wind = np.linspace(0, 25, 100)

plt.figure(figsize=(7, 4))
for inc in [25, 35, 45]:
    plt.plot(wind, toy_wind_backscatter(wind, inc), label=f'{inc}° incidence')
plt.xlabel('Wind speed (m/s)')
plt.ylabel('Toy backscatter (dB)')
plt.title('Wind Retrievals Must Account for Incidence Angle')
plt.grid(linestyle='--', linewidth=0.5, alpha=0.5)
plt.legend()
plt.show()

## Detecting Slicks and Oil-Like Features

Oil and some natural surface films damp short capillary waves. Because those waves contribute strongly to radar backscatter, slicks can appear as dark regions in SAR images.

A simple dark-object detection approach might:

1. Estimate the local background backscatter.
2. Identify pixels that are much darker than the background.
3. Remove very small detections.
4. Use shape, context, wind conditions, and ancillary data to distinguish oil from look-alikes.

Look-alikes are a major issue. Low wind, rain cells, biogenic slicks, current shadows, and atmospheric boundary-layer features can also produce dark SAR signatures.

In [ ]:
# Simple dark-feature detection on the synthetic SAR scene
scene_db = 10*np.log10(sigma0_filtered)
threshold_db = np.nanpercentile(scene_db, 12)
dark_mask = scene_db < threshold_db

plt.figure(figsize=(8, 4))
plt.imshow(dark_mask, origin='lower')
plt.title('Toy Dark-Feature Mask')
plt.xlabel('Range pixel')
plt.ylabel('Azimuth pixel')
plt.show()

print(f'Threshold used: {threshold_db:.2f} dB')
print(f'Fraction of scene flagged as dark: {dark_mask.mean()*100:.1f}%')

## Ships and Bright Targets

Ships often appear as bright targets because metal structures and sharp corners can return strong radar echoes. Ship detection commonly looks for local brightness anomalies relative to a surrounding background. This can be done with adaptive thresholding methods.

However, bright targets are not always ships. Offshore platforms, coastlines, sea ice ridges, rain cells, and processing artifacts can also be bright. Context is essential.

In [ ]:
# Add a few bright point targets to the synthetic scene
ship_scene = sigma0_filtered.copy()
ship_locations = [(55, 160), (100, 80), (125, 180)]
for row, col in ship_locations:
    ship_scene[row-1:row+2, col-1:col+2] += 0.5

ship_scene_db = 10*np.log10(ship_scene)
bright_threshold = np.nanpercentile(ship_scene_db, 99.8)
ship_candidates = ship_scene_db > bright_threshold

plt.figure(figsize=(8, 4))
plt.imshow(ship_scene_db, origin='lower')
plt.title('Toy SAR Scene with Bright Ship-Like Targets')
plt.xlabel('Range pixel')
plt.ylabel('Azimuth pixel')
cb = plt.colorbar()
cb.set_label('Backscatter (dB)')
plt.show()

print(f'Bright-target threshold: {bright_threshold:.2f} dB')
print(f'Candidate bright pixels: {ship_candidates.sum()}')

## Sea Ice Classification

Sea ice interpretation from SAR often uses combinations of backscatter level, texture, polarization, incidence angle, season, and prior ice information. SAR is valuable in polar regions because it can image through clouds and during polar night.

A simple classifier might separate open water, smooth young ice, and rougher deformed ice using thresholds and texture measures. Operational products usually require more careful calibration, expert interpretation, automated classification, and validation.

In [ ]:
# Toy sea ice classes using backscatter and local texture
np.random.seed(8)
water_db = np.random.normal(-18, 1.5, 400)
young_ice_db = np.random.normal(-13, 2.0, 400)
rough_ice_db = np.random.normal(-8, 3.0, 400)

plt.figure(figsize=(7, 4))
plt.hist(water_db, bins=25, alpha=0.5, label='Open water')
plt.hist(young_ice_db, bins=25, alpha=0.5, label='Young/smoother ice')
plt.hist(rough_ice_db, bins=25, alpha=0.5, label='Rougher ice')
plt.xlabel('Backscatter (dB)')
plt.ylabel('Count')
plt.title('Toy Backscatter Distributions for Sea Ice Interpretation')
plt.grid(linestyle='--', linewidth=0.5, alpha=0.5)
plt.legend()
plt.show()

## Summary

SAR ocean algorithms usually begin with calibrated backscatter and careful preprocessing. Important ideas include:

1. Calibrated backscatter allows more meaningful comparison across images.
2. Speckle is intrinsic to coherent radar imaging and must be managed.
3. Wind retrievals require geometry, polarization, radar frequency, and usually wind direction information.
4. Oil and natural slicks can appear dark because they damp short waves.
5. Ships often appear bright, but bright-target detection still requires context.
6. Sea ice products use backscatter, texture, polarization, season, and validation data.

# Check Your Understanding

1. Why are calibrated SAR products preferred for quantitative ocean applications?
2. What is speckle, and why is it common in SAR images?
3. Why does a SAR wind retrieval usually need incidence angle and wind direction information?
4. Name two look-alikes that can be confused with oil slicks in SAR imagery.
5. Why is SAR useful for sea ice mapping during polar night?

## References and Further Reading

- ESA Sentinel-1, *Oceans and ice* and Sentinel-1 application documentation.
- NOAA CoastWatch, *Monitoring Sea Surface Winds and Sea Ice with Satellite Radar*.
- NASA Earthdata/ARSET, *An Introduction to Synthetic Aperture Radar and Its Applications*.
- SAR Marine User's Manual.
- Sentinel-1 Level-2 Ocean product documentation.